In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_msgs(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]

    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
from langchain.agents import create_agent
from langchain.messages import trim_messages
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="claude-haiku-4-5", # switching to cheaper model in general
    checkpointer=InMemorySaver(),
    middleware=[trim_msgs]
)

In [8]:
from pprint import pprint
from langchain.messages import HumanMessage, AIMessage, ToolMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="device won't boot. what to do?"),
        ToolMessage("Diagnoser initiating diagnostic ping...", tool_call_id="1"),
        AIMessage(content="Make sure the device is plugged in and turned on."),
        HumanMessage(content="Yes, device is plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v ... complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What is the voltage?.")
    ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="device won't boot. what to do?", additional_kwargs={}, response_metadata={}, id='ada352be-5e1c-44c9-ae06-45fa20ad28d7'),
              AIMessage(content='Make sure the device is plugged in and turned on.', additional_kwargs={}, response_metadata={}, id='7f267d7a-37f5-406b-9e75-6105be3834e2', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content='Yes, device is plugged in and turned on.', additional_kwargs={}, response_metadata={}, id='a87aaab5-0949-4304-9f3d-c6fe096094f7'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='12ecbb58-b437-4708-9998-790dacebf3dd', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content='Yes, the red light is on.', additional_kwargs={}, response_metadata={}, id='54d17cbf-ccd6-4891-9a40-fbedd4e047a9'),
              AIMessage(content='A red light typically indicates a power or error state. Try these steps